# 🌌 OpenVista - Google Colab Edition

This notebook allows you to run the full OpenVista AI platform on Google Colab, leveraging a cloud GPU for image generation.

### 🛠️ Step 1: Setup Environment
Clone the repository and install all dependencies.

In [ ]:
# @title Clone and Bootstrap
import os
import sys
if not os.path.exists("OpenVista"):
    !git clone https://github.com/FayssalSabri/OpenVista.git

%cd /content/OpenVista
!bash scripts/colab_bootstrap.sh

# Explicitly install dependencies into the current environment
# Using sys.executable to ensure we install to the correct path
!{sys.executable} -m pip install -q -U celery fastapi uvicorn pyngrok nest-asyncio python-dotenv redis diffusers transformers accelerate

try:
    import celery
    print("✅ Celery is successfully installed!")
except ImportError:
    print("❌ Celery installation failed!")

### 🔑 Step 2: Configure Ngrok
To access the web interface, you need a free Ngrok token. 
1. Get your token here: [dashboard.ngrok.com](https://dashboard.ngrok.com/get-started/your-authtoken)
2. Paste it below.

In [ ]:
# @title Ngrok Auth
import os
NGROK_TOKEN = "" # @param {type:"string"}
os.environ["NGROK_TOKEN"] = NGROK_TOKEN

from pyngrok import ngrok
if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)
    # Kill existing sessions to avoid "already online" errors
    !pkill ngrok
else:
    print("❌ Please provide an NGROK_TOKEN!")

### 🚀 Step 3: Start Services
This will start the Backend (FastAPI), the AI Worker (Celery), and the Frontend (Next.js).

In [ ]:
# @title Run OpenVista
import subprocess
import time
import sys
import os
import threading
from pyngrok import ngrok
from google.colab import output

def log_reader(pipe, prefix):
    for line in iter(pipe.readline, b''):
        print(f"{prefix}: {line.decode().strip()}")

# 1. Setup Tunnels
print("📡 Setting up Tunnels...")
raw_backend_url = output.eval_js('google.colab.kernel.proxyPort(8000)')
backend_url = raw_backend_url.strip("/")
if not backend_url.startswith("http"):
    backend_url = f"https://{backend_url}"

try:
    ngrok.kill()
    frontend_tunnel = ngrok.connect(3000, name="openvista_frontend")
    frontend_url = frontend_tunnel.public_url
except Exception as e:
    print(f"⚠️ Ngrok failed ({e}), falling back to Colab Proxy...")
    raw_frontend_url = output.eval_js('google.colab.kernel.proxyPort(3000)')
    frontend_url = raw_frontend_url.strip("/")
    if not frontend_url.startswith("http"):
        frontend_url = f"https://{frontend_url}"

print(f"\n🔥 FRONTEND URL: {frontend_url}")
print(f"🔗 BACKEND URL: {backend_url}\n")

# 2. Update .env for Colab
with open(".env", "w") as f:
    f.write(f"REDIS_URL=redis://localhost:6379/0\n")
    f.write(f"NEXT_PUBLIC_API_URL={backend_url}\n")
    f.write(f"DATA_PATH=/content/OpenVista/data\n")
    f.write(f"MOCK_WORKER=false\n")
    f.write(f"FLUX_MODEL_ID=black-forest-labs/FLUX.1-schnell\n")

# 3. Prepare Environment Variables
custom_env = os.environ.copy()
# Add ALL subdirectories to path to ensure modules like 'worker' and 'pipeline' are found
backend_path = os.path.join(os.getcwd(), "backend")
custom_env["PYTHONPATH"] = f"{backend_path}:{custom_env.get('PYTHONPATH', '')}"

# 4. Start Backend
print("🟢 Starting Backend...")
backend_proc = subprocess.Popen([sys.executable, "-m", "uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"], 
                                cwd="backend", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=custom_env)
threading.Thread(target=log_reader, args=(backend_proc.stdout, "Backend"), daemon=True).start()

# 5. Start Worker
print("🟢 Starting Worker...")
worker_proc = subprocess.Popen([sys.executable, "-m", "celery", "-A", "worker", "worker", "--loglevel=info"], 
                               cwd="backend", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=custom_env)
threading.Thread(target=log_reader, args=(worker_proc.stdout, "Worker"), daemon=True).start()

# 6. Start Frontend
print("🟢 Starting Frontend (this might take 2-3 minutes)...")
# Remove --no-turbo as it is an unknown option, just use standard dev
frontend_proc = subprocess.Popen(["npm", "run", "dev", "--", "-p", "3000"], 
                                 cwd="frontend", stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
threading.Thread(target=log_reader, args=(frontend_proc.stdout, "Frontend"), daemon=True).start()

print("\n⌛ Waiting for services to initialize...")
time.sleep(10)
print("\n✨ MONITOR THE LOGS ABOVE. Once you see 'Ready' or 'Started', click the link:")
print(f"Open this link: {frontend_url}")

try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("🛑 Stopping services...")
    backend_proc.terminate()
    worker_proc.terminate()
    frontend_proc.terminate()
    ngrok.kill()